In [9]:
import pandas as pd
pd.reset_option('all')

import os, sys

import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt


# Calculating synergy scores using secondary screens as an example

In [10]:
cell_lines = ['B16', 'KPC', 'CT26', 'Renca']
conditions = []
for cell_line in cell_lines:
    conditions.append(cell_line+'_ICB_vs_'+cell_line+'_NSG')
conditions



['B16_ICB_vs_B16_NSG',
 'KPC_ICB_vs_KPC_NSG',
 'CT26_ICB_vs_CT26_NSG',
 'Renca_ICB_vs_Renca_NSG']

In [11]:
from functools import reduce




for condition in conditions:
    hypergeom = pd.read_table('../data/hypergeom/' + condition + '_volcano_frac100.txt', sep='\t', index_col=0)
    hypergeom = hypergeom[['Average LFC']]
    hypergeom = hypergeom.rename(columns={'Average LFC':'all_paralog_guides_avg_lfc'})
    
    hypergeom.to_csv('../data/hypergeom/'+condition+
                                '_top_paralog_subset_lfcs.txt', sep='\t')
        
    


In [12]:
def calc_synergy_score(paralog_lfcs, col, col_prefix):
    strong_scores = {}
    strong_scores['genes'] = []
    strong_scores[col_prefix+'_synergy_scores'] = []
    strong_scores[col_prefix+'_paralog_lfc'] = []
    strong_scores[col_prefix+'_first_gene_lfc'] = []
    strong_scores[col_prefix+'_second_gene_lfc'] = []


    for paralog, lfc in zip(paralog_lfcs.index, paralog_lfcs[col]):
        gene_one, _, gene_two, _ = paralog.split(';')
        gene_one_pair = ';'.join([gene_one, gene_one, 'ONE_INTERGENIC_SITE', 'ONE_INTERGENIC_SITE'])
        gene_two_pair = ';'.join([gene_two, gene_two, 'ONE_INTERGENIC_SITE', 'ONE_INTERGENIC_SITE'])
        gene_one_pair_lfc = list(lfcs[lfcs.index==gene_one_pair][col])[0]
        gene_two_pair_lfc = list(lfcs[lfcs.index==gene_two_pair][col])[0]
        indiv_argmax = np.argmax([np.abs(gene_one_pair_lfc), np.abs(gene_two_pair_lfc)])
        strong_score = lfc - [gene_one_pair_lfc, gene_two_pair_lfc][indiv_argmax]

        if np.sign(lfc) < 0:
            synergy = lfc - np.min([gene_one_pair_lfc,gene_two_pair_lfc,0])
            if synergy > 0:
                synergy=np.nan

        else:
            synergy = lfc - np.max([gene_one_pair_lfc,gene_two_pair_lfc,0])
            if synergy < 0:
                synergy=np.nan

        strong_scores['genes'].append(paralog)
        strong_scores[col_prefix+'_synergy_scores'].append(synergy)
        strong_scores[col_prefix+'_paralog_lfc'].append(lfc)
        strong_scores[col_prefix+'_first_gene_lfc'].append(gene_one_pair_lfc)
        strong_scores[col_prefix+'_second_gene_lfc'].append(gene_two_pair_lfc)
    strong_scores_df = pd.DataFrame(dict([(k,pd.Series(v)) for k,v in 
                                          strong_scores.items()]))
    strong_scores_df = strong_scores_df.set_index(['genes'])
    return strong_scores_df


In [13]:

for condition in conditions:
    lfcs = pd.read_csv('../data/hypergeom/'+condition+'_top_paralog_subset_lfcs.txt', sep='\t', index_col=0)
    paralogs = [x for x in lfcs.index if (('ONE_INTERGENIC_SITE' not in x) and ('NO_SITE' not in x))]
    paralog_lfcs = lfcs.loc[lfcs.index.isin(paralogs)]

    synergy_scores_dfs = []
    for col in paralog_lfcs.columns:
        num_guides = col.split('_')[0]
        synergy_scores_df = calc_synergy_score(paralog_lfcs,col,num_guides)
        synergy_scores_dfs.append(synergy_scores_df)

    synergy_scores_merged = reduce(lambda x, y: pd.merge(x, y, left_index=True, 
                                              right_index=True, how='inner'), synergy_scores_dfs)
    synergy_scores_merged = synergy_scores_merged[['all_synergy_scores', 'all_paralog_lfc', 
                                                   'all_first_gene_lfc', 'all_second_gene_lfc']]
    synergy_scores_merged = synergy_scores_merged.rename(columns={'all_first_gene_lfc':
                                                                  'first_gene_lfc',
                                                                  'all_second_gene_lfc':
                                                                  'second_gene_lfc'})

    synergy_scores_merged.to_csv('../data/synergy_scores/'+condition+'_synergy_scores_paralog_subsets.txt', sep='\t')


